In [ ]:
!uv pip install -U requests python-dotenv
PR_JSON_FILE = "prs.json"

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 6 packages in 170ms                                         
Audited 6 packages in 0.19ms


In [ ]:
# Get all PRs (title, date)
# Please add `GITHUB_TOKEN` to `.env` at the root of this repo

import os
import requests
import json

from pathlib import Path

from dotenv import load_dotenv

env_path = Path().resolve() / "../../.env"
load_dotenv(dotenv_path=env_path)

url = "https://api.github.com/repos/materialsproject/pymatgen/pulls"
params = {"state": "all", "per_page": 100, "page": 1}

# Setup GitHub token (for higher API rate limit)
if not (token := os.environ.get("GITHUB_TOKEN")):
    raise RuntimeError("Please set your GitHub token in GITHUB_TOKEN env var.")
headers = {"Authorization": f"Bearer {token}"}

# Load previous cache if exists
if os.path.exists(PR_JSON_FILE):
    print("Cache found, no fetch.")
else:
    pr_dict = {}  # {pr_number: {title, created_at, author}}

    while True:
        response = requests.get(url, headers=headers, params=params, timeout=60)
        response.raise_for_status()
        prs = response.json()
        if not prs:
            break
        for pr in prs:
            # Only keep merged PRs (comment out to keep all)
            if pr["merged_at"] is None:
                continue

            pr_dict[str(pr["number"])] = {
                "title": pr["title"],
                "created_at": pr["created_at"],
                "author": pr["user"]["login"],
            }
        params["page"] += 1

    with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(pr_dict, f, ensure_ascii=False)

    print(f"Saved {len(pr_dict)} PRs to {PR_JSON_FILE}")

Saved 2591 PRs to prs.json


In [ ]:
# Quickly clean up PRs (drop bot PRs and so on)
import json
import re

title_patterns = [
    r"(?i)pre[- ]?commit",
    r"(?i)dependabot",
]

author_patterns = [
    r"\[bot\]",
    r"(?i)pre[- ]?commit",
]

title_regexes = [re.compile(p) for p in title_patterns]
author_regexes = [re.compile(p) for p in author_patterns]


def match_any(regexes, text: str) -> bool:
    return any(r.search(text) for r in regexes)


with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)  # {number: {...}}

filtered = {}
uniq_dropped_titles = set()
uniq_dropped_authors = set()

for num, data in prs.items():
    title = (data.get("title") or "").strip()
    author = (data.get("author") or "").strip()

    if match_any(author_regexes, author):
        if author:
            uniq_dropped_authors.add(author)
        continue

    if match_any(title_regexes, title):
        if title:
            uniq_dropped_titles.add(title)
        continue

    filtered[num] = data

# Save cleaned
with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2, ensure_ascii=False)

print(f"Kept {len(filtered)} PRs. Saved to {PR_JSON_FILE}")
print(f"Dropped by title: (unique titles: {len(uniq_dropped_titles)})")
for t in sorted(uniq_dropped_titles):
    print("  •", (t[:120] + "…") if len(t) > 120 else t)

print(f"Dropped by author: (unique authors: {len(uniq_dropped_authors)})")
for a in sorted(uniq_dropped_authors):
    print("  •", a)

Kept 2445 PRs. Saved to prs.json
Dropped by title: (unique titles: 10)
  • Add `flake8` plugin `bugbear` as pre-commit hook
  • Add pydocstyle pre-commit hook
  • Add pylint pre-commit hook
  • Enable `dependabot` for `pip`
  • Enable flake8-simplify via ruff in pre-commit.ci
  • Fix pre-commit.ci isort error
  • Format aflow_prototypes.json and exclude it from codespell check in pre-commit
  • Improvement of pre-commit hook settings for `pylint`
  • chore: Included githubactions in the dependabot config
  • pre-commit.ci readme badge
Dropped by author: (unique authors: 3)
  • dependabot-preview[bot]
  • dependabot[bot]
  • pre-commit-ci[bot]


In [ ]:
# Feed PR titles to the LLM, looks like the context window for GPT-5 (400k)
# should be enough to feed all titles at once.
# ref: https://platform.openai.com/docs/models/gpt-5

# Dry run

import json
import math
from pathlib import Path
from collections import defaultdict


with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)

# Group titles by year
by_year = defaultdict(list)
for _, data in prs.items():
    title = (data.get("title") or "").strip()
    if not title:
        continue

    year: int = int((data.get("merged_at") or data.get("created_at"))[:4])
    by_year[year].append(title)

# Build prompt
instructions: str = """You are an expert technical summarizer.
I will give you GitHub pull request titles grouped by year.
For each year:
- Identify the 8-10 main technical topics.
- For each topic: provide a short label (2-5 words) and a 1-2 sentence description.
- Avoid personal names and issue numbers; focus on technical subject matter.
- Keep terminology consistent across years (same topic name for similar work).

Format your output as:

YEAR YYYY:
1. Topic label — description
...
"""

sections: list[str] = []
for year in sorted(by_year):
    titles = sorted(set(by_year[year]))  # de-dup within a year
    bullets = "\n".join(f"- {t}" for t in titles)
    sections.append(f"\nYEAR {year} PR titles:\n{bullets}")

prompt = instructions + "\n".join(sections)

# Rough size stats
tok_est: int = math.ceil(len(prompt) / 4)  # ~4 chars/token heuristic
print(f"Years: {len(by_year)}")
print(f"Total titles: {sum(len(v) for v in by_year.values())}")
print(f"Prompt length: {len(prompt):,} chars (~{tok_est:,} tokens est)")

# For inspection
print(prompt)

Years: 13
Total titles: 2445
Prompt length: 119,853 chars (~29,964 tokens est)
You are an expert technical summarizer.
I will give you GitHub pull request titles grouped by year.
For each year:
- Identify the 8-10 main technical topics.
- For each topic: provide a short label (2-5 words) and a 1-2 sentence description.
- Avoid personal names and issue numbers; focus on technical subject matter.
- Keep terminology consistent across years (same topic name for similar work).

Format your output as:

YEAR YYYY:
1. Topic label — description
...

YEAR 2013 PR titles:
- Add ntypat property to SiteCollection (raises AttributeError if not self.is_ordered)
- Add support for pickle to FloatWithUnit and ArrayWithUnit (unit tests included)
- ArrayWithUnit
- Better integration with Abinit (YAML format available in my Abinit developmental version)
- Better treatment of spurious data in HighSymmKpath
- Bug Fix for test_molecule_matcher
- Bug fix in the _spglib.c file shipped with pymatgen (see Yannick